# Clase 196 — Feast feature store

Definir features, generar training dataset point-in-time correct, materializar al online store, servir features con baja latencia.

Requiere: `pip install feast`.

## Setup

In [ ]:
import os, shutil, tempfile
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd, numpy as np

WORK = Path(tempfile.gettempdir()) / 'feast_demo'
if WORK.exists(): shutil.rmtree(WORK)
REPO = WORK / 'feature_repo'
(REPO / 'data').mkdir(parents=True)
os.chdir(WORK)
print('cwd:', Path.cwd())

## 1. Generar dataset histórico (offline store: parquet)

100 drivers × 30 días × 24 horas = 72 000 filas con features.

In [ ]:
rng = np.random.default_rng(42)
now = datetime(2026, 6, 1)
rows = []
for d_id in range(1001, 1101):
    for h in range(30 * 24):
        ts = now - timedelta(hours=h)
        rows.append({
            'driver_id': d_id,
            'event_timestamp': ts,
            'conv_rate': float(rng.beta(2, 5)),
            'acc_rate': float(rng.beta(8, 2)),
            'avg_daily_trips': int(rng.poisson(12)),
            'created': ts,
        })
df = pd.DataFrame(rows)
df.to_parquet(REPO / 'data' / 'driver_stats.parquet')
print(df.shape, '→', REPO / 'data' / 'driver_stats.parquet')

## 2. Definir el feature repo

`feature_store.yaml` (config) + `definitions.py` (entities, sources, feature views).

In [ ]:
(REPO / 'feature_store.yaml').write_text('''\
project: driver_demo
registry: data/registry.db
provider: local
online_store:
  type: sqlite
  path: data/online_store.db
offline_store:
  type: file
entity_key_serialization_version: 3
''')

(REPO / 'definitions.py').write_text('''\
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64

driver = Entity(name="driver", join_keys=["driver_id"])

src = FileSource(
    path="data/driver_stats.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

driver_stats_fv = FeatureView(
    name="driver_hourly_stats",
    entities=[driver],
    ttl=timedelta(days=7),
    schema=[
        Field(name="conv_rate", dtype=Float32),
        Field(name="acc_rate", dtype=Float32),
        Field(name="avg_daily_trips", dtype=Int64),
    ],
    source=src,
)
''')

os.chdir(REPO)
import subprocess
r = subprocess.run(['feast', 'apply'], capture_output=True, text=True)
print(r.stdout); print(r.stderr)

## 3. Training dataset point-in-time correct

Pedimos features para 3 drivers en 3 timestamps distintos. Feast devuelve el valor vigente en cada timestamp, no el más reciente.

In [ ]:
from feast import FeatureStore
store = FeatureStore(repo_path='.')

entity_df = pd.DataFrame({
    'driver_id': [1001, 1002, 1003],
    'event_timestamp': [
        datetime(2026, 5, 15, 10, 0),
        datetime(2026, 5, 20, 14, 0),
        datetime(2026, 5, 25, 8, 0),
    ],
    'label': [1, 0, 1],
})

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=['driver_hourly_stats:conv_rate', 'driver_hourly_stats:acc_rate', 'driver_hourly_stats:avg_daily_trips'],
).to_df()
training_df

## 4. Materializar al online store + servir features

Copiamos los valores más recientes de offline → online. Después consultamos por driver_id con latencia <2 ms.

In [ ]:
r = subprocess.run(['feast', 'materialize-incremental', '2026-06-01T23:00:00'], capture_output=True, text=True)
print(r.stdout[-400:]); print(r.stderr[-200:])

In [ ]:
online = store.get_online_features(
    features=['driver_hourly_stats:conv_rate', 'driver_hourly_stats:avg_daily_trips'],
    entity_rows=[{'driver_id': 1001}, {'driver_id': 1042}],
).to_dict()
print(online)

# Latencia
import time
t0 = time.perf_counter()
for _ in range(100):
    store.get_online_features(
        features=['driver_hourly_stats:conv_rate'],
        entity_rows=[{'driver_id': 1001}],
    ).to_dict()
print(f'latencia media: {(time.perf_counter() - t0) * 10:.2f} ms / request')

## Ejercicio guiado

1. Agregá una segunda entidad `merchant_id` y un `FeatureView` con `avg_rating`, `n_disputes_30d`.
2. Generá un training set con features de `driver` y `merchant` simultáneamente (join automático por Feast).
3. Bajá `ttl` a `timedelta(hours=1)` y verificá que `get_online_features` empieza a devolver `None` después de re-materializar con timestamp viejo.

## Conclusiones

- **Point-in-time joins** son lo que diferencia un feature store de un `LEFT JOIN`.
- El mismo código de cliente (`get_features`) sirve para training y para serving — ahí muere el training/serving skew.
- Online store local (SQLite) es para dev. En producción: Redis/DynamoDB.

## ✅ Soluciones de los ejercicios

Soluciones de los 5 ejercicios del README. Feast no está instalado, así que cada solución muestra el **comando/API real de Feast** (`feast init`, `get_historical_features`, `materialize-incremental`, `get_online_features`) y ejecuta el *concepto* con pandas: el corazón de una feature store es (1) el **point-in-time join** (nunca leer un valor del futuro) y (2) la dualidad **offline (histórico) / online (último valor por entidad con TTL)**. Ambos son reproducibles con `pandas.merge_asof`.

In [ ]:
import pandas as pd, numpy as np
from datetime import datetime, timedelta

# feature source "offline" (parquet en Feast): stats horarias por driver
base = datetime(2024, 1, 1, 8, 0)
rows = []
for driver in [1001, 1002]:
    for h in range(6):
        rows.append({'driver_id': driver,
                     'event_timestamp': base + timedelta(hours=h),
                     'conv_rate': round(0.5 + 0.05 * h + 0.1 * (driver == 1002), 3)})
feature_df = pd.DataFrame(rows).sort_values('event_timestamp')
print(feature_df.head())
print('\nCLI real: feast init driver_repo && feast apply && feast feature-views list')

### Ejercicio 1 — Setup mínimo

`feast init driver_repo` genera `feature_store.yaml` (config del provider/registry/online-store) y `example_repo.py` (define entities + FeatureViews). `feast apply` registra las definiciones. Mostramos la anatomía del `feature_store.yaml`.

In [ ]:
import yaml
feature_store_yaml = {
    'project': 'driver_repo',
    'registry': 'data/registry.db',
    'provider': 'local',
    'online_store': {'type': 'sqlite', 'path': 'data/online_store.db'},
    'entity_key_serialization_version': 2,
}
assert yaml.safe_load(yaml.safe_dump(feature_store_yaml)) == feature_store_yaml
print(yaml.safe_dump(feature_store_yaml, sort_keys=False))
print("feast apply -> registra Entity(driver_id) + FeatureView(driver_hourly_stats).")

### Ejercicio 2 — Training dataset histórico (point-in-time correct)

Este es EL concepto de una feature store offline: dado un `entity_df` con `(driver_id, event_timestamp)`, Feast devuelve para cada fila el **último valor del feature ANTERIOR o igual** a ese timestamp — nunca del futuro (eso sería *label leakage*). `pandas.merge_asof` con `direction='backward'` implementa exactamente eso.

In [ ]:
# API REAL de Feast:
#   store.get_historical_features(entity_df=entity_df,
#       features=["driver_hourly_stats:conv_rate"]).to_df()
entity_df = pd.DataFrame({
    'driver_id': [1001, 1001, 1001, 1002, 1002],
    'event_timestamp': [base + timedelta(hours=h) for h in [0.5, 2.5, 5.0, 1.0, 4.5]],
}).sort_values('event_timestamp')

training = pd.merge_asof(entity_df, feature_df, on='event_timestamp', by='driver_id',
                         direction='backward')
print(training.to_string(index=False))

# verificación manual: para driver 1001 @ t=+2.5h, el último valor <= t es el de h=2
row = training[(training.driver_id == 1001) &
               (training.event_timestamp == base + timedelta(hours=2.5))].iloc[0]
expected = feature_df[(feature_df.driver_id == 1001) &
                      (feature_df.event_timestamp == base + timedelta(hours=2))]['conv_rate'].iloc[0]
assert row['conv_rate'] == expected, 'point-in-time join tomó un valor del futuro (leakage!)'
print('\nOK — point-in-time correct: cada fila usa solo datos del pasado.')

### Ejercicio 3 — Materialización + serving online (latencia)

`feast materialize-incremental` empuja el **último** valor por entidad al online store (SQLite/Redis) para servir con latencia < 2 ms. Simulamos el online store como dict `driver_id -> último feature` y medimos el lookup.

In [ ]:
import time
# API REAL:
#   feast materialize-incremental $(date +%Y-%m-%d)
#   store.get_online_features(features=["driver_hourly_stats:conv_rate"],
#       entity_rows=[{"driver_id": 1001}]).to_dict()
online_store = (feature_df.sort_values('event_timestamp')
                .groupby('driver_id').last()['conv_rate'].to_dict())   # <- último por entidad
print('online store materializado:', online_store)

t0 = time.perf_counter()
for _ in range(10000):
    _ = online_store.get(1001)
lat_us = (time.perf_counter() - t0) / 10000 * 1e6
print(f'latencia lookup online: {lat_us:.2f} µs/req  (dict = O(1), como Redis)')
assert online_store[1001] == feature_df[feature_df.driver_id == 1001]['conv_rate'].iloc[-1]
print('OK — online = último valor por entidad; offline = historia completa.')

### Ejercicio 4 — TTL en acción

El `ttl` marca por cuánto tiempo un feature materializado sigue siendo válido. Si el último evento es más viejo que el TTL, `get_online_features` devuelve `None` (feature expirado). Lo implementamos.

In [ ]:
def get_online(driver_id, now, ttl: timedelta):
    sub = feature_df[feature_df.driver_id == driver_id]
    last_ts = sub['event_timestamp'].max()
    if now - last_ts > ttl:
        return None                      # expiró: fuera de TTL
    return sub.sort_values('event_timestamp')['conv_rate'].iloc[-1]

now = feature_df['event_timestamp'].max() + timedelta(days=3)   # pedimos 3 días después
assert get_online(1001, now, ttl=timedelta(days=1)) is None,  'con ttl=1d debe expirar'
assert get_online(1001, now, ttl=timedelta(days=7)) is not None, 'con ttl=7d sigue válido'
print('ttl=1d ->', get_online(1001, now, timedelta(days=1)), '(expirado)')
print('ttl=7d ->', get_online(1001, now, timedelta(days=7)), '(válido)')
print('OK — el TTL evita servir features rancios.')

### Ejercicio 5 — Skew check (offline vs online)

El *training/serving skew* aparece si el feature calculado offline (parquet) difiere del servido online (SQLite) para la misma entidad. Tras `materialize`, deben coincidir. Lo verificamos.

In [ ]:
skew = []
for driver_id in [1001, 1002]:
    offline_last = feature_df[feature_df.driver_id == driver_id].sort_values('event_timestamp')['conv_rate'].iloc[-1]
    online_val = online_store[driver_id]
    skew.append({'driver_id': driver_id, 'offline': offline_last,
                 'online': online_val, 'skew': abs(offline_last - online_val)})
skew_df = pd.DataFrame(skew)
print(skew_df.to_string(index=False))
assert (skew_df['skew'] == 0).all(), 'hay skew: la materialización quedó desincronizada (bug)'
print('\nOK — skew=0: online y offline consistentes tras materialize.')